# Error Analysis: Brute Force vs Smart Algorithm
## Comprehensive Comparison Across 10 Network Topologies

This notebook analyzes the performance differences between brute force and the optimized "smart" algorithm across 9 network topologies with detailed error percentage calculations and 3D visualizations.

In [ ]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns

# Load results
TOPOLOGY_DIR = r"C:\Users\ranik\Videos\files\topology_analysis"

topologies = [
    'erdos_renyi', 'barabasi_albert', 'watts_strogatz', 'path_graph', 'barbell',
    'star', 'tree', 'powerlaw_cluster', 'caveman'
]

def load_topology_results(topology_name):
    """Load results from a topology"""
    results_file = os.path.join(TOPOLOGY_DIR, topology_name, "results.json")
    if os.path.exists(results_file):
        with open(results_file, 'r') as f:
            return json.load(f)
    return None

print("✓ Libraries imported successfully")
print("✓ Loading results from all topologies...")

## Section 1: Summary Statistics

The analysis shows a clear divide between two categories of topologies:
- **Perfect Topologies (5 types)**: 0% error - Brute force and smart algorithm find same edges
- **Challenging Topologies (4 types)**: 15-85% error - Random/power-law graphs defeat heuristics

In [ ]:
def calculate_error_metrics(results):
    """Calculate comprehensive error metrics"""
    if not results or 'trials' not in results:
        return None
    
    trials_data = []
    
    for trial in results['trials']:
        bf_reduction = trial['brute_force']['reduction']
        sm_reduction = trial['smart']['reduction']
        
        # Error calculations
        absolute_error = bf_reduction - sm_reduction
        relative_error = (absolute_error / bf_reduction * 100) if bf_reduction > 0 else 0
        optimality = (sm_reduction / bf_reduction * 100) if bf_reduction > 0 else 0
        
        trials_data.append({
            'seed': trial['seed'],
            'nodes': trial['nodes'],
            'edges': trial['edges'],
            'bf_reduction': bf_reduction,
            'sm_reduction': sm_reduction,
            'absolute_error': absolute_error,
            'relative_error_pct': relative_error,
            'optimality_pct': optimality,
            'speedup': trial['metrics']['speedup'],
        })
    
    return pd.DataFrame(trials_data)

# Calculate metrics for all topologies
all_metrics = {}
for topology in topologies:
    results = load_topology_results(topology)
    df = calculate_error_metrics(results)
    if df is not None:
        all_metrics[topology] = df

print(f"✓ Loaded metrics for {len(all_metrics)} topologies")

## Section 2: Error Percentage Comparison Table